# VM1 후속 — hq 배치(≈17:30)가 끝난 뒤 (704daee 이후 판)
Square td·Can calql 사전학습은 VM1에서 더 이상 하지 않는다(`!pkill -f offline_pretrain.py`로 정리; hq의 train_dsrl.py는 영향 없음).
VM1은 hq 뒤에 **Can 온라인 6 run**을 맡는다: `calql_t12i`(교정이 온도가 잡힌 상태에서 사는가) ×3 + `calql_prefill`(동료 ①과 같은 데모 리플레이 설정) ×3, 150k. 둘 다 VM2가 만드는 `calql_can_s*.pt`(≈17:40)가 필요하다.
20:32 Drive 감사에서 이 6 run은 모두 미생성으로 확인됐다. 기존 환경이면 2 상태 확인 → 3 시작 → 4 keepalive. 새 VM이면 먼저 v3 설치 셀 0→1→2→3→5b→6→7→7b→8→9로 환경을 복원한다.

## 2. 상태 확인 — hq 7개 `[done]`, VM2의 `calql_can_s{1,2,3}.pt` 존재

In [ ]:
%%bash
PROJ=/content/drive/MyDrive/dsrl_project
for E in square_tent12_hq_s1 square_tent12_hq_s2 square_tent12_hq_s3 square_tent12_s4 square_tent12_s5 square_baseline_s4 square_baseline_s5; do
  echo "$E: $(grep '\[done\]\|\[eval\]' $PROJ/logs/$E.out | tail -n 1 | cut -c1-80)"; done
ls -lh $PROJ/logs/pretrain/calql_can_s*.pt 2>/dev/null
echo "processes: $(ps aux | grep -c '[t]rain_dsrl.py\|[o]ffline_pretrain.py')"; free -g | head -2

## 3. Can 온라인 6 run: `can_calql_t12i_s{1,2,3}` + `can_calql_prefill_s{1,2,3}`, 150k (RAM ≈ 102 GB). `calql_can_s*.pt`가 없으면 그 run은 건너뛰고 나중에 다시 실행

In [ ]:

%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && source /content/env.sh && cd /content/dsrl
git pull origin o2o | tail -n 1
PROJ=/content/drive/MyDrive/dsrl_project
CFG="--config-path=cfg/robomimic --config-name=dsrl_can.yaml"
BASE="log_dir=$PROJ/logs train.total_env_steps=150000"
launch () { EXP=$1; shift; nohup python train_dsrl.py $CFG exp_id=$EXP "$@" > $PROJ/logs/$EXP.out 2>&1 & echo "started $EXP (pid $!)"; }
for S in 1 2 3; do
  PT=$PROJ/logs/pretrain/calql_can_s$S.pt
  test -f $PT || { echo "calql_can_s$S.pt 아직 없음 (VM2 사전학습 뒤 다시)"; continue; }
  launch can_calql_t12i_s$S    seed=$S variant=calql pretrain_path=$PT $BASE offline_mix.mode=none load_offline_data=False train.ent_coef=auto_0.3 train.target_ent=12
  launch can_calql_prefill_s$S seed=$S variant=calql pretrain_path=$PT $BASE offline_mix.mode=prefill offline_data_path=$PROJ/offline/can_train_offline.npz
done
sleep 150; for E in can_calql_t12i_s1 can_calql_prefill_s1; do echo "== $E"; grep "\[pretrain\]\|\[eval\]\|offline\|Error\|Traceback" $PROJ/logs/$E.out | tail -n 3; done; free -g | head -2


## 3. keepalive (반납 없음)

In [ ]:
import subprocess, time
PROJ = '/content/drive/MyDrive/dsrl_project'
def sh(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
while True:
    running = sh("ps aux | grep '[o]ffline_pretrain.py\\|[t]rain_dsrl.py' | grep -o 'exp_id=[a-z_0-9]*\\|pretrain.method=[a-z]*\\|seed=[0-9]*' | tr '\\n' ' '")
    ram = sh("free -g | awk 'NR==2{print $3\"/\"$2}'")
    prog = sh("for f in $(ls -t %s/logs/pretrain_*.out %s/logs/can_*.out %s/logs/square_*.out 2>/dev/null | head -n 12); do "
              "n=$(basename $f .out); l=$(grep '^\\[cql\\]\\|^\\[calql\\]\\|^\\[distill\\]\\|\\[eval\\]\\|\\[done\\]' $f | tail -n 1 | cut -c1-70); "
              "echo -n \"$n: $l | \"; done" % (PROJ, PROJ, PROJ))
    print(time.strftime('%H:%M'), 'ram', ram, '|', running or '(none running)', '|', prog, flush=True)
    if not running:
        print('nothing running (VM kept)', flush=True)
        break
    time.sleep(600)

## 5. (사용 안 함) Square calql 온라인은 VM3의 14번이 cql과 함께 띄운다

In [ ]:
# (비워 둠)
